# TFIM Hardware Demo — Qiskit Fall Fest 2026
Trotterized Transverse-Field Ising Model: transpilation levels, best-qubit layout, ZNE, Aer + IBM hardware.
Runs in Colab or local `qff26` venv. No hardcoded tokens (uses `getpass` / `.env`).

In [ ]:
# Colab only (skip locally):
# !pip install "qiskit[visualization]==2.5.2" "qiskit-aer==0.17.2" "qiskit-ibm-runtime==0.50.0" "scipy==1.18.1" "matplotlib==3.11.2" "pandas==3.0.6"

import os, json, sys
sys.path.insert(0, ".")
import hardware as hw
print("hardware.py loaded,", "qiskit:", hw._QISKIT_AVAILABLE)

## 1. Backend setup (PINQ2 shared allocation)

In [ ]:
from getpass import getpass
USE_HARDWARE = bool(os.environ.get("QISKIT_IBM_TOKEN") or os.environ.get("PINQ2_TOKEN"))
svc = backend = binfo = None
if USE_HARDWARE:
    svc = hw.get_service(channel="ibm_quantum_platform")  # reads .env / env
    backend = hw.select_backend(svc, preferred=("ibm_marrakesh", "ibm_fez"), min_qubits=4)
    binfo = hw.print_backend_properties(backend)
else:
    print("No token -> Aer-only mode. Set QISKIT_IBM_TOKEN to enable hardware.")
    # token = getpass("Paste PINQ2 token (leave empty for Aer-only): ")

## 2. TFIM circuit (N=4, J=1, h=0.5, dt=0.1, steps=3)

In [ ]:
N, J, h, DT, STEPS, SHOTS, SEED = 4, 1.0, 0.5, 0.1, 3, 4096, 42
circ_meas = hw.build_tfim_circuit(N, J, h, DT, STEPS, add_measure=True)
circ = hw.build_tfim_circuit(N, J, h, DT, STEPS, add_measure=False)
obs = hw.tfim_observables(N)
print(circ_meas)
print("observables:", list(obs))

## 3. Transpilation comparison (levels 0-3)

In [ ]:
rows = hw.transpile_comparison(circ_meas, backend, levels=(0,1,2,3), seed=SEED)
hw.print_comparison_table(rows)

## 4. Hardware-aware layout (best qubits)

In [ ]:
if backend is not None:
    t_hw, meta = hw.hardware_aware_transpile(circ_meas, backend, N, seed=SEED)
    print("best_qubits:", meta["best_qubits"], "depth:", meta["depth"], "est_error:", meta["est_error"])
else:
    print("Skipped (no backend). Best-qubit selection needs backend.target/coupling_map.")

## 5-6. Aer execution + ZNE

In [ ]:
exact = hw.exact_magnetization(N, J, h, DT, STEPS)
print("exact magnetization:", exact)
aer_val = hw.aer_estimator_value(circ, obs["magnetization"], shots=SHOTS, seed=SEED)
print("Aer estimator:", aer_val)
zne = hw.run_zne_aer(circ, obs["magnetization"], shots=SHOTS, seed=SEED)
print("ZNE raw:", zne["raw"], "mitigated:", zne["mitigated"])
counts = hw.run_aer_counts(circ_meas, shots=SHOTS, seed=SEED)
print("Aer counts mag:", hw.counts_magnetization(counts, N))

## 6b. Real hardware (only if token present; expect queue)

In [ ]:
if svc is not None and backend is not None:
    bname = getattr(backend, "name", "")
    out = hw.run_hardware(svc, bname, circ, {"magnetization": obs["magnetization"]}, shots=1024)
    print(json.dumps({k: v for k, v in out.items() if k != "counts"}, indent=2, default=str))
else:
    print("Hardware run skipped (Aer-only).")

## 7. Full pipeline: JSON + PNGs

In [ ]:
payload = hw.run_pipeline(n=N, J=J, h=h, dt=DT, steps=STEPS, shots=SHOTS, seed=SEED,
                          backend_name=getattr(backend, "name", "") if backend else "",
                          outdir="results", no_hardware=(backend is None))
print("saved -> results/tfim_results.json + 3 PNGs")